# 강화 학습 (Reinforcement learning) 3

In [1]:
import random 
import math 

import matplotlib.pyplot as plt
import numpy as np

import torch
from torch import nn, optim
import torch.nn.functional as F
from collections import deque 

import gymnasium as gym 

ModuleNotFoundError: No module named 'gymnasium'

## Q learning : Frozen Lake

In [ ]:
import gym
import numpy as np
import matplotlib.pyplot as plt
from gym.envs.registration import register
import random

def rargmax(vector):
    m = np.amax(vector)
    indices = np.nonzero(vector == m)[0]
    return random.choice(indices)
# a= [0, 1, 1, 0]
# np.amax(a)
# b = np.nonzero(np.array(a) == 1)[0]
# print(b)
# random.choice(b)

register(
    id='FrozenLake-v3',
    entry_point = 'gym.envs.toy_text:FrozenLakeEnv',
    kwargs={'map_name':'4x4',
           'is_slippery':False}
)

env = gym.make('FrozenLake-v3')

# Q Table을 모두 0으로 초기화 한다. : 2차원 (number of state, action space) = (16,4)
Q = np.zeros([env.observation_space.n, env.action_space.n])

# 몇 번 시도를 할 것인가 (에피소드)
num_episodes = 2000

# 에피소드마다 총 리워드의 합을 저장하는 리스트
rList = []

for i in range(num_episodes) : 
    state = env.reset() # (0, {'prob': 1})
    
    # newer versions of gym 
    if isinstance(state, tuple):
        state = state[0]
    rAll = 0
    done = False
    
    # Q learning 알고리즘
    while not done : 
        # Action 중에 가장 R(Reward)이 큰 Action을 고른다. 
        # 이 때, random noise 방식으로 decaying Exploit & Exploration 구현 
        
        action = rargmax(Q[state, :])
        
        # 해당 Action을 했을 때 environment가 변하고, 새로운 state, reward, done 여부를 반환 받음
        # state, reward, done, truncated, info
        new_state, reward, done, truncated, info = env.step(action)        
        Q[state, action] = reward + np.max(Q[new_state, :])
                     
        rAll += reward
        state = new_state
        
    rList.append(rAll)

print("Success rate : "+str(sum(rList) / num_episodes))
    
print("Final Q-Table Values")
print(Q)

plt.figure(figsize = (15, 5) )
plt.bar(range(len(rList)), rList, color="blue")
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Reward per Episode')
plt.show()



## Q learning_greedy_discount : Frozen Lake

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

def rargmax(vector):
    m = np.amax(vector)
    indices = np.nonzero(vector == m)[0]
    return random.choice(indices)

env = gym.make('FrozenLake-v1', desc=None,
                map_name="4x4", is_slippery=False)

# Q Table을 모두 0으로 초기화 한다. : 2차원 (number of state, action space) = (16,4)
Q = np.zeros([env.observation_space.n, env.action_space.n])

# 몇 번 시도를 할 것인가 (에피소드)
num_episodes = 2000
gamma = 0.99
# 에피소드마다 총 리워드의 합을 저장하는 리스트
rList = []

for i in tqdm(range(num_episodes)) : 
    state, _ = env.reset()
    
    # decaying E-greedy
    e = 1./((i/100) + 1)

    # newer versions of gym 
    if isinstance(state, tuple):
        state = state[0]
    rAll = 0
    done = False
    
    # Q learning 알고리즘
    while not done : 
        # Action 중에 가장 R(Reward)이 큰 Action을 고른다. 
        # 이 때, random noise 방식으로 decaying Exploit & Exploration 구현 
        # E-greedy

        if np.random.rand(1) < e:
            action = env.action_space.sample() # exploration
        else:
            action = rargmax(Q[state, :]) # exploit

        
        ## Random noise
        # action = np.argmax(Q[state, :] + np.random.randn(1, env.action_space.n)/(i+1))
        # action = rargmax(Q[state, :] + np.random.randn(env.action_space.n)/(i+1)) # dimension correction

        # 해당 Action을 했을 때 environment가 변하고, 새로운 state, reward, done 여부를 반환 받음
        # state, reward, done, truncated, info
        new_state, reward, done, truncated, info = env.step(action)        
        Q[state, action] = reward + gamma*np.max(Q[new_state, :])
                     
        rAll += reward
        state = new_state
        
    rList.append(rAll)

print("Success rate : "+str(sum(rList) / num_episodes))
    
print("Final Q-Table Values")
print(Q)

plt.bar(range(len(rList)), rList, color="blue")
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Reward per Episode')
plt.show()



## Q learning_stochastic : Frozen Lake

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import random
from tqdm import tqdm

def rargmax(vector):
    m = np.amax(vector)
    indices = np.nonzero(vector == m)[0]
    return random.choice(indices)

env = gym.make('FrozenLake-v1', desc=None,
                map_name="4x4", is_slippery=False)

# Q Table을 모두 0으로 초기화 한다. : 2차원 (number of state, action space) = (16,4)
Q = np.zeros([env.observation_space.n, env.action_space.n])

# 몇 번 시도를 할 것인가 (에피소드)
num_episodes = 20000
gamma = 0.99 # decay
lr = 0.1 # learning rate
# 에피소드마다 총 리워드의 합을 저장하는 리스트
rList = []

for i in tqdm(range(num_episodes)) : 
    state = env.reset()
    
    e = 1./((i/100) + 1)
    # newer versions of gym 
    if isinstance(state, tuple):
        state = state[0]
    rAll = 0
    done = False
    
    # Q learning 알고리즘
    while not done : 
        # Action 중에 가장 R(Reward)이 큰 Action을 고른다. 
        # 이 때, random noise 방식으로 decaying Exploit & Exploration 구현 
        # E-greedy
        if np.random.rand(1) < e:
            action = env.action_space.sample() # exploration
        else:
            action = rargmax(Q[state, :]) # exploit

        
        ## Random noise
        # action = np.argmax(Q[state, :] + np.random.randn(1, env.action_space.n)/(i+1))
        # action = rargmax(Q[state, :] + np.random.randn(env.action_space.n)/(i+1)) # dimension correction

        # 해당 Action을 했을 때 environment가 변하고, 새로운 state, reward, done 여부를 반환 받음
        # state, reward, done, truncated, info
        new_state, reward, done, truncated, info = env.step(action)        
        Q[state, action] = (1-lr)*Q[state, action]+ lr*(reward + gamma*np.max(Q[new_state, :]))
                     
        rAll += reward
        state = new_state
        
    rList.append(rAll)

print("Success rate : "+str(sum(rList) / num_episodes))
    
print("Final Q-Table Values")
print(Q)

plt.bar(range(len(rList)), rList, color="blue")
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Reward per Episode')
plt.show()